<a href="https://colab.research.google.com/github/tahumada/MSO-AEON/blob/main/DECam/DECam_submit_list2AEON.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DECam target submission to the AEON queue
Tomas Ahumada - tomas.ahumada@noirlab.edu

Last modified: Aug 2026

This code intends to be a tutrorial for NOIRLab-AEON users of the Dark Energy Camera (DECam) mounted at the 4m V. M. Blanco telescope. The AEON queue is run by the Las Cumbres Observatory (LCO) Scheduler, thus the user requires an active account to access the LCO portal and generate a LCO key to submit requests to an active program.

Information about DECam can be found here: https://noirlab.edu/science/programs/ctio/instruments/Dark-Energy-Camera
Information about LCO can be found here: https://observe.lco.global/

Once you have an active user in the LCO portal, you can find the API key here https://observe.lco.global/accounts/profile

# Outline
1. Read example target list
2. Get template request (json format) - this json file is modified and later sent to the LCO queue
3. Make the payload, modifying the json template.
4. Submit targets individually. Each target can have multiple filters.

In [17]:
import copy
from datetime import datetime, timedelta
import glob
import io
import json
import os
import re
import time
import urllib.request
from urllib.parse import urlparse
from astropy.time import Time
import numpy as np
import pandas as pd
import requests

In [18]:


# Base raw URL
raw_url = "https://raw.githubusercontent.com/tahumada/MSO-AEON/main/DECam/example_list_targets_decam"

# 1. Append timestamp cache-buster query parameter
url_no_cache = f"{raw_url}?_={int(time.time())}"

# 2. Force HTTP headers to prevent intermediary/CDN caching
headers = {
    'Cache-Control': 'no-cache, no-store, must-revalidate',
    'Pragma': 'no-cache',
    'Expires': '0'
}

response = requests.get(url_no_cache, headers=headers)
response.raise_for_status()
targets = pd.read_csv(io.StringIO(response.text))

targets


,sourceID,ra,dec,filters,exptime
0,ZTF25acfxvcu,0.646663,-3.710376,g r i,30 30 30
1,ZTF24aatlsjq,1.277942,29.962589,g r i,30 60 90
2,ZTF25abqvssg,2.797111,1.804104,g r z,30 120 90
3,ZTF24aaymkrs,3.089388,31.063407,r z,30 60
4,ZTF24aawklme,3.090119,17.794190,i,90
5,ZTF25abjjlgl,5.021151,8.511636,g r i z,30 150 30 300
6,ZTF24aboafrj,5.466447,9.255386,g z,300 300
7,ZTF25aaqqjud,5.723418,46.144067,g i,60 60


In [19]:
# get example json
def get_example(example_path):
    # Automatically convert standard GitHub file URLs to raw GitHub URLs
    if "github.com" in example_path and "/blob/" in example_path:
        example_path = example_path.replace("github.com", "raw.githubusercontent.com").replace("/blob/", "/")

    # Check if the string is a URL
    if urlparse(example_path).scheme in ('http', 'https'):
        response = requests.get(example_path)

        if response.status_code == 200:
            return response.json()
        else:
            raise Exception(f"Failed to fetch file from URL. Status code: {response.status_code}")
    else:
        # Code to handle local file paths...
        pass



In [21]:
template_json_url = "https://raw.githubusercontent.com/tahumada/MSO-AEON/main/example.json"
print('The template of an observation:')
get_example(template_json_url)

The template of an observation:


{'id': 2495241,
 'requests': [{'id': 4168585,
   'location': {'telescope_class': '4m0'},
   'configurations': [{'id': 14811742,
     'constraints': {'max_airmass': 1.6,
      'min_lunar_distance': 30.0,
      'max_lunar_phase': 1.0,
      'max_seeing': None,
      'min_transparency': None,
      'extra_params': {}},
     'instrument_configs': [{'exposure_time': 40.0,
       'optical_elements': {'filter': 'r'},
       'mode': 'default',
       'exposure_count': 1,
       'rotator_mode': '',
       'extra_params': {'offset_ra': 0, 'offset_dec': 0}}],
     'acquisition_config': {'mode': 'OFF',
      'exposure_time': None,
      'extra_params': {}},
     'guiding_config': {'optional': True,
      'mode': 'ON',
      'optical_elements': {},
      'exposure_time': None,
      'extra_params': {}},
     'target': {'type': 'ICRS',
      'name': 'test2',
      'ra': 151.197,
      'dec': 7.801,
      'proper_motion_ra': 0.0,
      'proper_motion_dec': 0.0,
      'parallax': 0.0,
      'epoch': 2

In [ ]:
# make payload

def make_payload(variables, template_path):
    data = get_example(template_path)

    # 1. Base constraints and configurations
    req_config = data['requests'][0]['configurations'][0]

    req_config['constraints']['max_airmass'] = variables['maximum_airmass']
    req_config['constraints']['minimum_lunar_distance'] = variables['minimum_lunar_distance']

    # Safely formatting the date (zero-padding months and days)
    now = Time.now().datetime
    date = f"{now.year}{now.month:02d}{now.day:02d}"

    data['name'] = f"{variables['target']['id']}_{date}"
    data['proposal'] = variables['PROPOSAL_ID']
    data['requests'][0]['windows'] = variables['windows']

    req_config['target']['name'] = variables['target']['id']
    req_config['target']['ra'] = str(variables['target']['ra'])
    req_config['target']['dec'] = str(variables['target']['dec'])

    # centering
    req_config['extra_params']['detector_centering'] = 'S4'

    # base instrument configuration template
    base_instrument_config = req_config['instrument_configs'][0]

    # Reset filter configuration in the payload
    req_config['instrument_configs'] = []

    # loop through filters, copy, and append
    for filt in ['g', 'r', 'i', 'z']:
        if filt in variables['target']['filters']:
            # Create a completely independent copy of the dictionary
            new_config = copy.deepcopy(base_instrument_config)

            new_config['optical_elements']['filter'] = filt
            idx = list(variables['target']['filters']).index(filt)
            new_config['exposure_time'] = float(variables['target']['exptime'][idx])
            req_config['instrument_configs'].append(new_config)

    return data

In [ ]:
LCO_TOKEN ='d9b9d2faf4803e4bfd8efe803a2a943e629ca8fc'
requestpath  = "https://observe.lco.global/api/requestgroups/"

PROPOSAL_ID  = '2013A-9999'
WINDOW_START = '2026-08-01 17:32:00'
WINDOW_END   = '2026-09-01 17:32:00'

# saving the status of the request
lco_sent,lco_failed = [],[]

# change this if you do not want to submit the request, and just see how the payload looks
send = True

for i in range(min(2, len(targets))):

    name, ra, dec, filters, exptime = targets.iloc[i][['sourceID', 'ra', 'dec', 'filters','exptime']]
    filter_list = filters.split(' ')
    exptime_list = exptime.split(' ')

    variables = {
        "PROPOSAL_ID": PROPOSAL_ID,
        'maximum_airmass': 1.4,
        'minimum_lunar_distance': 20,
        'windows': [{'start': WINDOW_START, 'end': WINDOW_END}],
        'target': {'id': name,
                   'ra': ra,
                   'dec': dec,
                   'filters': filter_list,
                   'exptime': exptime_list
                   }
    }

    print(variables)

    data = make_payload(variables,template_path = template_json_url)
    print(data)

    # send as test
    if send:
      response = requests.post(
              requestpath,
              headers={"Authorization": f"Token {LCO_TOKEN}"},
              json=data,  # Make sure you use json!
          )

      if response.status_code == 400:
          print(variables['target']['id'], 'Failed (sending to queue)')
          lco_failed.append([variables['target']['id'],response.text])

      elif response.status_code == 201 or response.status_code == 200:
          lco_sent.append([variables['target']['id'],'sent!',response.json()['id']])



{'PROPOSAL_ID': '2013A-9999', 'maximum_airmass': 1.4, 'minimum_lunar_distance': 20, 'windows': [{'start': '2026-08-01 17:32:00', 'end': '2026-09-01 17:32:00'}], 'target': {'id': 'ZTF25acfxvcu', 'ra': np.float64(0.646663), 'dec': np.float64(-3.710376), 'filters': ['g', 'r', 'i'], 'exptime': ['30', '30', '30']}}
{'id': 2495241, 'requests': [{'id': 4168585, 'location': {'telescope_class': '4m0'}, 'configurations': [{'id': 14811742, 'constraints': {'max_airmass': 1.4, 'min_lunar_distance': 30.0, 'max_lunar_phase': 1.0, 'max_seeing': None, 'min_transparency': None, 'extra_params': {}, 'minimum_lunar_distance': 20}, 'instrument_configs': [{'exposure_time': 30.0, 'optical_elements': {'filter': 'g'}, 'mode': 'default', 'exposure_count': 1, 'rotator_mode': '', 'extra_params': {'offset_ra': 0, 'offset_dec': 0}}, {'exposure_time': 30.0, 'optical_elements': {'filter': 'r'}, 'mode': 'default', 'exposure_count': 1, 'rotator_mode': '', 'extra_params': {'offset_ra': 0, 'offset_dec': 0}}, {'exposure_ti

In [ ]:
print('sources sent:', lco_sent)


sources sent: [['ZTF25acfxvcu', 'sent!', 2644908]]


In [ ]:
print('sources failed:',lco_failed)

sources failed: [['ZTF24aatlsjq', '{"requests":[{"non_field_errors":["According to the constraints of the request, the target will not be visible within the time window. Check that the target is in the nighttime sky. Consider modifying the time window or loosening the airmass or lunar separation constraints. If the target is non sidereal, double check that the provided elements are correct."]}]}']]


In [ ]:
ids = np.asarray(lco_sent).T[-1]
print(ids)
for idlco in ids:
    response = requests.post(
                f'https://observe.lco.global/api/requestgroups/{idlco}/cancel/',
                headers={"Authorization": f"Token {LCO_TOKEN}"},
            )

    if response.status_code == 200:
      print('request:',idlco,'cancelled')

['2644908']
request: 2644908 cancelled
